# Engineering of Data Analysis - lecture 2

Engineering of Data Analysis is an elective course of the MSc in Business Analytics of NOVA SBE. The course assumes knowledge of Python programming language and Pandas framework.

This notebook presents examples and exercises of lecture 2.


In [4]:
import pandas as pd
import numpy as np


# Libraries are our friends

Python is a high level language. A Python program is interpreted by a Python interpreter, in which the program is executed line by line.

There are some techniques to optimize the execution of a Python program, by compiling it to a Bytecode and running this bytecode in a Python virtual machine or even compiling it to machine code.

## Python and numpy

Let's compare the speed of executing simple operations using Python and the same operation in a Python library.

In [5]:
array = np.random.rand(10**7)


In [6]:
%%timeit

total_sum_code = 0.0
for number in array:
    total_sum_code += number



1.42 s ± 416 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
%%timeit

total_sum_sum = sum(array)

1.28 s ± 357 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit

total_sum_lib = array.sum()



7.79 ms ± 455 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### What do these results tell us?

The three versions compute the same sum, but at very different speeds:

| Version | Typical time | Relative speed |
|---|---|---|
| Python `for` loop | ~1.5 s | 1× (baseline) |
| Python built-in `sum()` | ~1.3 s | ~1× |
| NumPy `array.sum()` | ~8 ms | **~180× faster** |

**Why such a large difference?**

As explained in the slides (*CPUs and Python*, slide 7), Python is an *interpreted* language: the Python interpreter processes each line of code at runtime, adding significant overhead per operation. A simple multiplication inside a Python loop can cost 20–100 clock cycles just for the interpreter overhead — on top of the actual computation.

NumPy functions like `array.sum()` are implemented in compiled C code that runs directly on the CPU, with no interpreter overhead and with SIMD vectorisation (processing multiple values per clock cycle). This matches what the slides show: **use library functions whenever possible** (slide 9).

Note that `sum(array)` is almost as slow as the `for` loop: Python's built-in `sum` iterates element by element in Python, converting each NumPy scalar to a Python float on every step — the same interpreter overhead applies.

Let's check the three versions get the same value.

In [9]:
total_sum_lib = array.sum()
print( total_sum_lib)

total_sum_sum = sum(array)
print( total_sum_sum)

total_sum_code = 0.0
for number in array:
    total_sum_code += number
print( total_sum_code)


5000271.365649524
5000271.36564925
5000271.36564925


**Why are the values not exactly equal?**

NumPy's `array.sum()` accumulates floating-point numbers internally in 64-bit precision, using pairwise summation to reduce rounding error. Python's `sum()` and the `for` loop convert each NumPy scalar to a Python `float` object before adding — this conversion introduces small rounding errors that accumulate across 10 million additions.

This is a consequence of interpreter overhead (slide 7): even the *type conversion* between NumPy's internal representation and Python's object model has a numerical cost, not just a speed cost.

The value is not exactly the same because when executing Python based sums, the values are rounded when they are converted from the CPU representation to the Python representation.

## Python and Pandas

Let's try to do the same using Pandas.

In [10]:
df = pd.DataFrame( {'val' : array})
df

,val
0,0.514337
1,0.709149
2,0.484357
3,0.629754
4,0.695839
...,...
9999995,0.151375
9999996,0.910759
9999997,0.740823
9999998,0.985976


In [11]:
%%timeit
df.agg({'val': 'sum'})

18.2 ms ± 920 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [12]:
%%timeit
df['val'].sum()

17.6 ms ± 925 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### What do these results tell us?

| Version | Typical time | Relative speed |
|---|---|---|
| `df['val'].sum()` | ~18 ms | 1× (baseline) |
| `df.agg({'val': 'sum'})` | ~18 ms | ~1× |
| `df['val'].agg(lambda x: x.sum())` | ~540 ms | **~30× slower** |

**Why is Pandas fast — and why does the custom function break it?**

Pandas operations like `.sum()` and `.agg('sum')` delegate to the same compiled NumPy routines under the hood — so they are fast for the same reason as `array.sum()`: no Python interpreter overhead per element.

The `lambda x: x.sum()` version is different: Pandas must call back into Python to execute the lambda, which reintroduces interpreter overhead. For a 10-million-row array, this means 10 million Python function call roundtrips.

This distinction matters for the map/reduce model (introduced later in this lecture): **built-in aggregations** (min, max, mean, sum) are implemented in compiled code and can be parallelised automatically by frameworks like Spark and cuDF. **Custom `apply` functions** are Python callbacks — they are portable but cannot be optimised further by the framework.

Let's check when using a custom function.

In [13]:
%%timeit
df['val'].agg( lambda x: x.sum())

543 ms ± 6.12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Memory and Python

In a computer, data is stored on disks for persistency and in memory for being accessed.

Additionally, CPUs have several levels of caches, for speeding up access to data. The following examples show how caching can influence the speed of programs.


### What do these results tell us?

Both loops sum the same 10 million values (`1000 × 10000`), but in different orders:

| Access pattern | Typical time | Relative speed |
|---|---|---|
| Row-major: `arraydouble[i][j]` (outer=row) | ~930 ms | **~1.5× faster** |
| Column-major: `arraydouble[i][j]` (outer=column) | ~1.5 s | 1× (baseline) |

**Why does access order matter?**

As explained in the slides (*Memory, disk and caches*, slides 10–11 and *Memory and Python*, slide 13), CPUs do not fetch individual values from RAM — they fetch **cache lines** (typically 64 bytes, i.e. 8 consecutive 64-bit floats) at a time. When you access `arraydouble[i][j]` with `j` in the inner loop, the next element `arraydouble[i][j+1]` is already in the cache line just fetched — a **cache hit**, essentially free.

When `i` is in the inner loop, the next element `arraydouble[i+1][j]` is 10,000 positions away in memory — the cache line just fetched is useless, and a new one must be loaded from RAM (**cache miss**). With RAM access taking 60–100 ns vs cache access at 1–40 ns (slide 11), the cost adds up across 10 million iterations.

The key takeaway from the slides: **data locality matters**. Libraries like NumPy and Pandas store data in contiguous row-major arrays (C order) precisely to exploit cache lines during row-wise operations.

In [14]:
arraydouble = [[i * 100 + j for j in range(10000)] for i in range(10000)]


In [15]:
%%timeit

sum_arraydouble = 0
for i in range(1000):
    for j in range(10000):
        sum_arraydouble += arraydouble[i][j]


926 ms ± 266 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
%%timeit

sum_arraydouble = 0
for j in range(10000):
    for i in range(1000):
        sum_arraydouble += arraydouble[i][j]


1.46 s ± 277 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
del arraydouble

### What do these results tell us?

Both loops sum 1 million values from a 1-billion-element array, but at different strides:

| Access pattern | Stride | Typical time | Relative speed |
|---|---|---|---|
| Every element: `bigarray[i]` | 1 | ~210 ms | **~1.3× faster** |
| Every 1000th element: `bigarray[i*1000]` | 1000 | ~275 ms | 1× (baseline) |

The difference is smaller than in the 2D example above, because both patterns access memory sequentially enough that hardware prefetchers partially compensate. But the cost of strided access is still measurable.

**What this shows:**

When stride = 1, consecutive accesses fall within the same cache line — the CPU prefetcher can detect the pattern and fetch ahead. When stride = 1000 (elements 8000 bytes apart), each access lands in a different cache line in a region spread across 8 GB of RAM. The array is 8 GB total — far too large for any cache (slide 11: caches are 1–64 MB). Every access is a cache miss and requires a fetch from RAM.

This is the same principle behind the slide 13 takeaway: **improve data locality using contiguous data structures**. A 10-million-row Pandas DataFrame of 10 float columns is ~800 MB — it fits in RAM but not in cache, so access patterns within Pandas operations have the same kind of impact on real workloads.

Let's do it with a different example.

In [18]:
bigarray = np.random.rand(10**9)

In [19]:
%%timeit

total_sum_1 = 0
for i in range(0,10**6,1):
    total_sum_1 += bigarray[i]


209 ms ± 37.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [20]:
%%timeit

total_sum_100 = 0
for i in range(0,1000*10**6,1000):
    total_sum_100 += bigarray[i]


273 ms ± 5.52 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
# Let's free the memory of bigarray
del bigarray

### Connection to the lecture

The examples above demonstrate the memory hierarchy from the slides in action. Sequential memory access keeps data in the CPU cache (fast); strided or scattered access causes cache misses and forces data to be fetched from RAM (slow). This is the same principle that makes data locality important for parallel hardware — keeping data close to the processor that needs it.